In [ ]:
!pip install -q transformers datasets torchaudio librosa scikit-learn accelerate torchcodec

In [ ]:
import os
import random
import torch
import torchaudio
import numpy as np

from torch.utils.data import Dataset

from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    DataCollatorWithPadding,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

In [ ]:
# =========================================
# CONFIGURACIÓN GENERAL
# =========================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATASET_PATH = "/content/MESD"

SAMPLE_RATE = 16000
NUM_LABELS = 3
SEED = 42
K_FOLDS = 5

In [ ]:
# =========================================
# SEMILLA
# =========================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# =========================================
# MAPEO DE EMOCIONES
# =========================================

emotion2id = {
    "happiness": 0,
    "neutral": 1,
    "sadness": 2,
    "fear": 2,
    "surprise": 2,
    "anger": 2,
    "disgust": 2
}

id2emotion = {
    0: "positivo",
    1: "neutral",
    2: "negativo"
}


In [ ]:
# =========================================
# DATASET
# =========================================

class EmotionDataset(Dataset):

    def __init__(self, files, data_dir, processor):

        self.files = files
        self.data_dir = data_dir
        self.processor = processor

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        filename = self.files[idx]

        path = os.path.join(self.data_dir, filename)

        # Obtener emoción desde el nombre
        emotion = filename.split("_")[0].lower()

        label = emotion2id[emotion]

        # =====================================
        # CARGA AUDIO
        # =====================================

        speech, sr = torchaudio.load(path)

        # Resample
        if sr != SAMPLE_RATE:

            speech = torchaudio.transforms.Resample(
                sr,
                SAMPLE_RATE
            )(speech)

        # Convertir a mono
        speech = speech.mean(dim=0)

        MIN_SECONDS = 1
        MIN_SAMPLES = SAMPLE_RATE * MIN_SECONDS

        if len(speech) < MIN_SAMPLES:

            padding = MIN_SAMPLES - len(speech)

            speech = torch.nn.functional.pad(
                speech,
                (0, padding)
            )

        # =====================================
        # PROCESAMIENTO
        # =====================================

        inputs = self.processor(
            speech.numpy(),
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            padding=True
        )

        return {
            "input_values": inputs.input_values.squeeze(),
            "labels": torch.tensor(label)
        }


In [ ]:
# =========================================
# MODELO Y PROCESSOR
# =========================================

MODEL_NAME = "jonatasgrosman/wav2vec2-large-xlsr-53-spanish"

processor = Wav2Vec2Processor.from_pretrained(
    MODEL_NAME
)

data_collator = DataCollatorWithPadding(
    tokenizer=processor,
    padding=True,
    return_tensors="pt"
)

In [ ]:
# =========================================
# CARGAR ARCHIVOS
# =========================================

MESD_all_files = [
    f for f in os.listdir(DATASET_PATH)
    if f.endswith(".wav")
]

In [ ]:
# =========================================
# ETIQUETAS PARA ESTRATIFICACIÓN
# =========================================

MESD_all_labels = []

for file in MESD_all_files:

    emotion = file.split("_")[0].lower()

    label = emotion2id[emotion]

    MESD_all_labels.append(label)

In [ ]:
# =========================================
# DIVISIÓN DEL DATASET
# 80% TRAIN + VALIDATION
# 20% TEST FINAL
# =========================================

MESD_train_val_files, MESD_test_files, MESD_train_val_labels, MESD_test_labels = train_test_split(
   MESD_all_files,
    MESD_all_labels,
    test_size=0.2,
    random_state=SEED,
    stratify=MESD_all_labels
)

print("\n==========================")
print("DIVISIÓN DEL DATASET")
print("==========================")

print(f"Train+Validation: {len(MESD_train_val_files)}")
print(f"Test Final: {len(MESD_test_files)}")

In [ ]:
# =========================================
# MÉTRICAS
# =========================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1-score": f1
    }

In [ ]:
# =========================================
# BASELINES
# =========================================

#def random_baseline(y_true, num_labels):

 #   preds = np.random.randint(
  #      0,
   #     num_labels,
    #    size=len(y_true)
    #)

    #precision, recall, f1, _ = precision_recall_fscore_support(
     #   y_true,
      #  preds,
       # average="macro",
        #zero_division=0
  #  )

    #acc = accuracy_score(y_true, preds)

    #return {
     #   "accuracy": acc,
      #  "precision": precision,
       # "recall": recall,
        #"f1-score": f1
    #}


#def majority_baseline(y_true):

    # Clase más frecuente
 #   majority_class = max(
  #      set(y_true),
   #     key=y_true.count
    #)

    #preds = [majority_class] * len(y_true)

    #precision, recall, f1, _ = precision_recall_fscore_support(
     #   y_true,
      #  preds,
       # average="macro",
        #zero_division=0
    #)

    #acc = accuracy_score(y_true, preds)

    #return {
     #   "accuracy": acc,
      #  "precision": precision,
       # "recall": recall,
        #"f1-score": f1
    #}

In [ ]:
# =========================================
# ARGUMENTOS DE ENTRENAMIENTO
# =========================================

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=8,
    warmup_steps=10,
    logging_dir="./logs",
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="f1-score",
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    seed=SEED
)

In [ ]:
# =========================================
# STRATIFIED K-FOLD
# =========================================

skf = StratifiedKFold(
    n_splits=K_FOLDS,
    shuffle=True,
    random_state=SEED
)

In [ ]:


# =========================================
# CROSS VALIDATION
# =========================================

cv_results = []

for fold, (MESD_train_idx, MESD_test_idx) in enumerate(
    skf.split(MESD_train_val_files, MESD_train_val_labels)
):

    print("\n==========================")
    print(f"FOLD {fold+1}/{K_FOLDS}")
    print("==========================")

    # =====================================
    # SPLITS DEL FOLD
    # =====================================

    fold_train_files = [
        MESD_train_val_files[i]
        for i in MESD_train_idx
    ]
#test
    fold_val_files = [
        MESD_train_val_files[i]
        for i in MESD_test_idx
    ]

    # =====================================
    # DATASETS
    # =====================================

    MESD_train_dataset = EmotionDataset(
        fold_train_files,
        DATASET_PATH,
        processor
    )

    MESD_val_dataset = EmotionDataset(
        fold_val_files,
        DATASET_PATH,
        processor
    )

    # =====================================
    # REINICIAR MODELO
    # =====================================

    model = Wav2Vec2ForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        mask_time_prob=0.0
    ).to(DEVICE)

    # Congelar extractor
    for param in model.wav2vec2.feature_extractor.parameters():
        param.requires_grad = False

    # =====================================
    # TRAINER
    # =====================================

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=MESD_train_dataset,
        eval_dataset=MESD_val_dataset,
        processing_class=processor,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    # =====================================
    # ENTRENAMIENTO
    # =====================================

    trainer.train()

    # =====================================
    # EVALUACIÓN DEL FOLD
    # =====================================

    results = trainer.evaluate(MESD_test_idx)

    print("\nResultados Fold:")

    for k, v in results.items():

        if isinstance(v, float):
            print(f"{k}: {v:.4f}")

    cv_results.append(results)

In [ ]:


# =========================================
# RESULTADOS PROMEDIO CV
# =========================================

print("\n==========================")
print("RESULTADOS PROMEDIO CV")
print("==========================")

metricas = [
    "eval_accuracy",
    "eval_precision",
    "eval_recall",
    "eval_f1-score"
]

for metrica in metricas:

    valores = [r[metrica] for r in cv_results]

    promedio = np.mean(valores)

    desviacion = np.std(valores)

    print(
        f"{metrica}: "
        f"{promedio:.4f} ± {desviacion:.4f}"
    )

In [ ]:
# =========================================
# BASELINES TEST FINAL
# =========================================

# Etiquetas reales del test
#y_true = []

#for item in test_dataset:
   # y_true.append(item["labels"].item())

# =========================================
# RANDOM BASELINE
# =========================================

#random_results = random_baseline(
  #  y_true,
   # NUM_LABELS
#)

#print("\n==========================")
#print("BASELINE RANDOM")
#print("==========================")

#for k, v in random_results.items():
 #   print(f"{k}: {v:.4f}")

# =========================================
# MAJORITY BASELINE
# =========================================

#majority_results = majority_baseline(
 #   y_true
#)

#print("\n==========================")
#print("BASELINE MAJORITY")
#print("==========================")

#for k, v in majority_results.items():
 #   print(f"{k}: {v:.4f}")

SyntaxError: unmatched ')' (3780859296.py, line 525)